# BNN ile Talep Belirsizliği ve Stok/Üretim Optimizasyonu

Bu notebook, **Pyro ile Bayesçi Sinir Ağı (BNN)** kurup posterior predictive talep senaryolarını **Pyomo + HiGHS** ile bir stokastik üretim kararına bağlar.

Akış: veri → BNN → posterior predictive → senaryolar → Sample Average Approximation (SAA) → karar.

Tek dönemlik problem:
\[
\min_x c_p x+\frac1S\sum_s[c_h(x-D_s)^+ + c_u(D_s-x)^+].
\]

In [ ]:
# Gerekirse:
# %pip install torch pyro-ppl numpy pandas matplotlib pyomo highspy

import random, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
import pyro, pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.nn import PyroModule, PyroSample
import pyomo.environ as pyo

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); pyro.set_rng_seed(SEED)
torch.set_default_dtype(torch.float32)

## 1. Sentetik talep verisi

Gerçek uygulamada bu bölüm ERP/MES/IoT veya pazar verileriyle değiştirilir. Özellikler promosyon, sıcaklık, sezon ve trenddir.

In [ ]:
n=220
day=np.arange(n)
promotion=np.random.binomial(1,0.25,n)
temperature=20+9*np.sin(2*np.pi*day/60)+np.random.normal(0,2,n)
season=np.sin(2*np.pi*day/30)
trend=day/(n-1)
true_mean=105+15*season+22*promotion+0.55*temperature+10*trend+7*promotion*season
demand=true_mean+np.random.normal(0,8,n)

df=pd.DataFrame({"day":day,"promotion":promotion,"temperature":temperature,
                 "season":season,"trend":trend,"demand":demand})
df.head()

In [ ]:
features=["promotion","temperature","season","trend"]
X_np=df[features].to_numpy(np.float32); y_np=df["demand"].to_numpy(np.float32)
X_mean=X_np.mean(0,keepdims=True); X_std=X_np.std(0,keepdims=True)+1e-6
y_mean=float(y_np.mean()); y_std=float(y_np.std()+1e-6)
X=torch.tensor((X_np-X_mean)/X_std)
y=torch.tensor((y_np-y_mean)/y_std)

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(day,demand,label="Gözlenen talep")
ax.plot(day,true_mean,label="Gerçek koşullu ortalama")
ax.set_xlabel("Gün"); ax.set_ylabel("Talep"); ax.legend(); plt.show()

## 2. Pyro ile BNN

Standart ağda \(w=\hat w\); BNN'de \(w\sim p(w)\). Veri sonrası yaklaşık posterior:
\[
q_\phi(w)\approx p(w\mid\mathcal D).
\]
`AutoDiagonalNormal` hızlı bir mean-field variational posterior sağlar. Risk-kritik projelerde kalibrasyon ayrıca test edilmelidir.

In [ ]:
class BayesianDemandNN(PyroModule):
    def __init__(self,in_features,hidden=16):
        super().__init__()
        self.hidden=PyroModule[nn.Linear](in_features,hidden)
        self.hidden.weight=PyroSample(dist.Normal(0,1).expand([hidden,in_features]).to_event(2))
        self.hidden.bias=PyroSample(dist.Normal(0,1).expand([hidden]).to_event(1))
        self.out=PyroModule[nn.Linear](hidden,1)
        self.out.weight=PyroSample(dist.Normal(0,1).expand([1,hidden]).to_event(2))
        self.out.bias=PyroSample(dist.Normal(0,1).expand([1]).to_event(1))
    def forward(self,x,y=None):
        mean=self.out(torch.tanh(self.hidden(x))).squeeze(-1)
        sigma=pyro.sample("sigma",dist.LogNormal(-1.0,0.35))
        with pyro.plate("data",x.shape[0]):
            pyro.sample("obs",dist.Normal(mean,sigma),obs=y)
        return mean

pyro.clear_param_store()
model=BayesianDemandNN(X.shape[1])
guide=AutoDiagonalNormal(model)
svi=SVI(model,guide,pyro.optim.Adam({"lr":0.015}),loss=Trace_ELBO())

losses=[]
for step in range(2500):
    losses.append(svi.step(X,y)/len(y))
    if (step+1)%500==0:
        print(f"Adım {step+1}: ELBO/gözlem={losses[-1]:.4f}")

plt.plot(losses); plt.xlabel("SVI adımı"); plt.ylabel("ELBO/gözlem"); plt.show()

## 3. Posterior predictive talep senaryoları

`obs` örnekleri ağırlık belirsizliği ile gözlem gürültüsünü birlikte taşır. Böylece BNN bir **senaryo üreticisi** olur.

In [ ]:
future=np.array([[1.0,27.0,0.85,1.05]],dtype=np.float32)
future_X=torch.tensor((future-X_mean)/X_std)

predictive=Predictive(model,guide=guide,num_samples=2000,return_sites=("obs","_RETURN"))
samples=predictive(future_X)
demand_samples=samples["obs"].detach().cpu().numpy().reshape(-1)*y_std+y_mean
mean_samples=samples["_RETURN"].detach().cpu().numpy().reshape(-1)*y_std+y_mean

print(pd.Series(demand_samples).describe(percentiles=[.05,.5,.95]))
plt.hist(demand_samples,bins=40,density=True,alpha=.7)
plt.axvline(np.mean(demand_samples),linestyle="--",label="Ortalama")
plt.axvline(np.quantile(demand_samples,.95),linestyle=":",label="%95 quantile")
plt.xlabel("Talep"); plt.ylabel("Yoğunluk"); plt.legend(); plt.show()

## 4. Pyomo ile stokastik optimizasyon

BNN'den \(S\) senaryo seçiyoruz. `excess` ve `shortage` değişkenleri pozitif parça ifadelerini lineerleştirir.

Bu ayrım önemlidir: **BNN belirsizliği modeller; OR modeli bu belirsizlik altında kararı seçer.**

In [ ]:
rng=np.random.default_rng(SEED)
S=500
scenario=np.clip(rng.choice(demand_samples,S,replace=False),0,None)

production_cost=2.0
holding_cost=1.0
shortage_cost=7.0
capacity=180.0

m=pyo.ConcreteModel()
m.S=pyo.RangeSet(0,S-1)
m.x=pyo.Var(bounds=(0,capacity))
m.excess=pyo.Var(m.S,domain=pyo.NonNegativeReals)
m.shortage=pyo.Var(m.S,domain=pyo.NonNegativeReals)
D={s:float(scenario[s]) for s in range(S)}

m.excess_con=pyo.Constraint(m.S,rule=lambda M,s: M.excess[s]>=M.x-D[s])
m.shortage_con=pyo.Constraint(m.S,rule=lambda M,s: M.shortage[s]>=D[s]-M.x)
m.obj=pyo.Objective(
    expr=production_cost*m.x+(1/S)*sum(
        holding_cost*m.excess[s]+shortage_cost*m.shortage[s] for s in m.S),
    sense=pyo.minimize)

result=pyo.SolverFactory("appsi_highs").solve(m)
optimal_x=pyo.value(m.x)
print("Optimum üretim:",round(optimal_x,2))
print("Beklenen amaç değeri:",round(pyo.value(m.obj),2))
print("Durum:",result.solver.termination_condition)

## 5. Deterministik ortalama-talep kararıyla karşılaştırma

Tahmin doğruluğu tek başına yeterli değildir. Endüstri mühendisliğinde asıl değerlendirme **downstream decision quality** üzerinden yapılmalıdır.

In [ ]:
deterministic_x=min(float(np.mean(demand_samples)),capacity)

def realized_cost(x,d):
    return production_cost*x+holding_cost*max(x-d,0)+shortage_cost*max(d-x,0)

cost_bnn=np.array([realized_cost(optimal_x,d) for d in demand_samples])
cost_det=np.array([realized_cost(deterministic_x,d) for d in demand_samples])

comparison=pd.DataFrame({
    "Karar":["BNN + stokastik optimizasyon","Ortalama talep"],
    "Üretim":[optimal_x,deterministic_x],
    "Beklenen maliyet":[cost_bnn.mean(),cost_det.mean()],
    "Maliyet %95 quantile":[np.quantile(cost_bnn,.95),np.quantile(cost_det,.95)],
    "Stokout olasılığı":[np.mean(demand_samples>optimal_x),np.mean(demand_samples>deterministic_x)]
})
comparison

## 6. Risk ölçüsü: CVaR

Beklenen maliyet yerine kuyruk riskini izlemek için:
\[
\operatorname{CVaR}_{\alpha}(L)
=\eta+\frac{1}{1-\alpha}\mathbb E[(L-\eta)^+].
\]
Tam CVaR optimizasyonunda \(\eta\) ve fazlalık değişkenleri Pyomo modeline doğrudan eklenebilir.

In [ ]:
def empirical_cvar(costs,alpha=.95):
    var=np.quantile(costs,alpha)
    return float(costs[costs>=var].mean())

print("BNN+SAA CVaR95:",round(empirical_cvar(cost_bnn),2))
print("Deterministik CVaR95:",round(empirical_cvar(cost_det),2))

## 7. Endüstri mühendisliğine genişletme

| BNN'in tahmin ettiği belirsizlik | OR problemi |
|---|---|
| Talep | stok / üretim planlama |
| İşlem süresi | job-shop / flow-shop çizelgeleme |
| Lead time | tedarik zinciri |
| Arıza / kalan faydalı ömür | bakım planlama |
| Taşıma süresi | VRP / filo planlama |
| Enerji tüketimi / kalite | proses optimizasyonu |
| Pahalı simülasyon çıktısı | simulation optimization |

Gerçek projede ayrıca **kalibrasyon**, OOD davranışı, Deep Ensemble/GP baseline'ları, senaryo sayısı duyarlılığı, CVaR/chance constraints ve model drift test edilmelidir.

### Kullanılan araçlar
- PyTorch: sinir ağı altyapısı
- Pyro: BNN + variational inference
- Pyomo: matematiksel programlama
- HiGHS (`highspy`): açık kaynak LP/MIP çözücü
- NumPy / pandas / Matplotlib: veri ve görselleştirme